# Домашнее задание 9. Панельные модели экспорта и поиск недоосвоенных рынков

**Оценивание.** `Неделя 9_НС_Поиск экспортных рынков_ДЗ.ipynb` весит 50% оценки. PDF-отчёт также весит 50% оценки. Максимальный балл за домашнее задание составляет 10 баллов.

**Пояснение к разбалловке.** За все пункты синего уровня можно получить максимум 5 баллов из 10. Пункты синего и красного уровней дают максимум 7 баллов из 10. Пункты всех трёх уровней дают максимум 10 баллов из 10. Пункт 3г даёт 1 бонусный балл чёрного уровня, но итоговый балл за домашнее задание всё равно не может превышать 10. Дополнительная идея с кластеризацией и расстоянием в конце задания выполняется только по желанию и не оценивается.


In [1]:
# @title Служебная загрузка данных — нажми ▶ { display-mode: "form" }
# Запусти эту ячейку перед выполнением заданий. Это данность -- как и в семинаре 9,
# разбираться в реализации не обязательно, важно знать, что возвращает каждая функция.
!pip install linearmodels
import requests
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats
pd.options.display.float_format = "{:,.3f}".format

OEC_BASE_URL = "https://api-v2.oec.world/tesseract"
TRADE_CUBE = "trade_i_baci_a_17"

def load_oec_members(level, search=None, limit="100,0", cube=TRADE_CUBE):
    params = {"cube": cube, "level": level, "limit": limit}
    if search is not None:
        params["search"] = search
    response = requests.get(f"{OEC_BASE_URL}/members.jsonrecords", params=params, timeout=60)
    response.raise_for_status()
    return pd.DataFrame(response.json()["members"]), response.url

def load_oec_data(drilldowns, include=None, measures="Trade Value", cube=TRADE_CUBE, limit=50000):
    params = {"cube": cube, "drilldowns": ",".join(drilldowns), "measures": measures,
              "locale": "en", "parents": "false", "limit": f"{limit},0"}
    if include:
        params["include"] = ";".join(include)
    response = requests.get(f"{OEC_BASE_URL}/data.jsonrecords", params=params, timeout=120)
    response.raise_for_status()
    payload = response.json()
    table = pd.DataFrame(payload["data"])
    if table.empty:
        raise ValueError("OEC вернул пустую таблицу: проверь код товара, страну-экспортёра и годы.")
    return table, payload.get("annotations", {}), response.url

def load_wdi(indicator, start_year, end_year):
    response = requests.get(f"https://api.worldbank.org/v2/country/all/indicator/{indicator}",
        params={"format": "json", "date": f"{start_year}:{end_year}", "per_page": 20000}, timeout=120)
    response.raise_for_status()
    payload = response.json()
    if not payload or len(payload) < 2 or payload[1] is None:
        raise ValueError(f"World Bank API вернул пустой ответ для индикатора {indicator}.")
    table = pd.DataFrame({"ISO3": r["countryiso3code"], "Year": int(r["date"]), "Значение": r["value"]}
                          for r in payload[1] if r["countryiso3code"] and r["value"] is not None)
    return table, response.url

def find_country_id(country_name, level="Exporter Country", cube=TRADE_CUBE):
    # Ищет ID страны в справочнике cube по названию, например 'Russia'. Схема ответа OEC:
    # столбцы 'key' (технический ID) и 'caption' (человекочитаемое название).
    members, _ = load_oec_members(level, search=country_name, cube=cube)
    if members.empty:
        raise ValueError(f"Страна '{country_name}' не найдена по уровню '{level}', уточни запрос.")
    match = members.loc[members["caption"].str.contains(country_name, case=False, na=False)]
    row = match.iloc[0] if not match.empty else members.iloc[0]
    return row["key"], row

def find_product_id(product_search, product_level="HS6", name_fragment=None, cube=TRADE_CUBE):
    # Ищет ID товара в справочнике cube (уровень HS6/HS4/...) по свободному текстовому запросу.
    # ВАЖНО: 'key' в OEC для товаров -- это НЕ голый HS-код. Для мороженого (HS6 210500) реальный
    # key равен '4210500': это код товарной секции (4 -- Prepared Foodstuffs) + сам HS6-код.
    # Поэтому товар всегда ищем через members-справочник, а не подставляем HS-код напрямую.
    members, _ = load_oec_members(product_level, search=product_search, cube=cube)
    if members.empty:
        raise ValueError(f"Товар '{product_search}' не найден на уровне {product_level}, уточни запрос.")
    if name_fragment:
        match = members.loc[members["caption"].str.contains(name_fragment, case=False, na=False)]
        row = match.iloc[0] if not match.empty else members.iloc[0]
    else:
        row = members.iloc[0]
    return row["key"], row

def load_bilateral_exports(exporter_name, product_search, product_level="HS6", name_fragment=None,
                            start_year=2016, end_year=2023, cube=TRADE_CUBE):
    # Экспорт товара (найденного по product_search на уровне product_level) из страны exporter_name
    # по всем странам-партнёрам за start_year..end_year. Возвращает панель (Year, ISO3, Страна, Exports).
    exporter_id, _ = find_country_id(exporter_name, cube=cube)
    product_id, _ = find_product_id(product_search, product_level=product_level,
                                     name_fragment=name_fragment, cube=cube)
    years = ",".join(str(y) for y in range(start_year, end_year + 1))
    include = [f"{product_level}:{product_id}", f"Exporter Country:{exporter_id}", f"Year:{years}"]
    raw, meta, url = load_oec_data(["Year", "Importer Country", product_level], include, cube=cube)
    raw["ISO3"] = raw["Importer Country ID"].str[-3:].str.upper()
    panel = (raw.groupby(["Year", "ISO3", "Importer Country"], as_index=False)["Trade Value"].sum()
                .rename(columns={"Importer Country": "Страна", "Trade Value": "Exports"}))
    return panel.sort_values(["ISO3", "Year"]), {"dataset_name": meta.get("dataset_name"), "url": url}

def attach_wdi(panel, indicator, col_name, start_year=2016, end_year=2023):
    # Присоединяет к панели (со столбцами ISO3, Year) один показатель Всемирного банка.
    wdi, url = load_wdi(indicator, start_year, end_year)
    wdi = wdi.rename(columns={"Значение": col_name})
    return panel.merge(wdi[["ISO3", "Year", col_name]], on=["ISO3", "Year"], how="left"), url

# Товар команды: фанера, HS4 4412, как на неделях 7 и 8.
# BACI в кубе HS17 начинается с 2018 года, поэтому берём 2018–2024.
EXPORTER = "Russia"
PRODUCT_SEARCH, PRODUCT_LEVEL, PRODUCT_NAME_FRAGMENT = "plywood", "HS4", "Plywood"
START_YEAR, END_YEAR = 2018, 2024
import warnings
warnings.filterwarnings("ignore")
print("Загрузчики готовы.")


Загрузчики готовы.


## **[Синий, 5 баллов]** Задание 1. Панель данных и pooled OLS

### **[Синий, 1 балл]** Пункт 1а. Структура панели

Загрузи панель экспорта товара из России по странам-импортёрам и проверь её на дубликаты, число стран и лет, сбалансированность.


In [2]:
# Загрузи панель экспорта товара из России и проверь её структуру.
trade_panel, meta = load_bilateral_exports(
    EXPORTER, PRODUCT_SEARCH, PRODUCT_LEVEL, PRODUCT_NAME_FRAGMENT, START_YEAR, END_YEAR
)
product_key, product_row = find_product_id(PRODUCT_SEARCH, PRODUCT_LEVEL, PRODUCT_NAME_FRAGMENT)
print(f"Товар: {product_row['caption']}, HS4 4412 (ключ OEC {product_key}), экспортёр: {EXPORTER}")
print(f"Источник: {meta['dataset_name']}")

# Выведи период, число стран, число строк, дубликаты и число лет на страну.
years = sorted(trade_panel["Year"].unique())
n_years = len(years)
duplicates = trade_panel.duplicated(["ISO3", "Year"]).sum()
years_per_country = trade_panel.groupby("ISO3")["Year"].nunique()

print(f"Период: {years[0]}–{years[-1]} ({n_years} лет)")
print(f"Стран: {trade_panel['ISO3'].nunique()}, строк: {len(trade_panel)}")
print(f"Дубликатов пары страна–год: {duplicates}")
print(f"Нулевых или отрицательных потоков: {(trade_panel['Exports'] <= 0).sum()}")
print(f"Стран со всеми {n_years} годами: {(years_per_country == n_years).sum()}, "
      f"панель сбалансирована: {years_per_country.eq(n_years).all()}")
print(f"Стран с одним наблюдением: {(years_per_country == 1).sum()}")

display(years_per_country.value_counts().sort_index().rename_axis("Лет у страны").to_frame("Стран"))

by_year = trade_panel.groupby("Year").agg(
    Стран=("ISO3", "size"),
    Экспорт_млн_долл=("Exports", lambda values: values.sum() / 1e6),
)
display(by_year)

# Откуда несбалансированность: страны, которые пропали из панели после 2021 года
last_year = trade_panel.groupby("ISO3")["Year"].max()
first_year = trade_panel.groupby("ISO3")["Year"].min()
left_after_2021 = last_year[last_year <= 2022].index
names = trade_panel.drop_duplicates("ISO3").set_index("ISO3")["Страна"]
print(f"Последний год потока 2022 или раньше: {len(left_after_2021)} стран, например: "
      f"{', '.join(names[left_after_2021[:12]])}")
print(f"Появились в панели позже {years[0]} года: {(first_year > years[0]).sum()} стран")


Товар: Plywood, HS4 4412 (ключ OEC 94412), экспортёр: Russia
Источник: HS6 REV. 2017 (2018 - 2024)
Период: 2018–2024 (7 лет)
Стран: 141, строк: 701
Дубликатов пары страна–год: 0
Нулевых или отрицательных потоков: 0
Стран со всеми 7 годами: 48, панель сбалансирована: False
Стран с одним наблюдением: 11


,Стран
Лет у страны,
1,11
2,13
3,7
4,18
5,29
6,15
7,48


,Стран,Экспорт_млн_долл
Year,,
2018,117,"1,446.642"
2019,125,"1,237.854"
2020,119,"1,216.678"
2021,120,"2,004.763"
2022,95,"1,513.717"
2023,72,821.083
2024,53,730.946


Последний год потока 2022 или раньше: 68 стран, например: Aruba, Afghanistan, Australia, Belgium, Benin, Bangladesh, Bulgaria, Belarus, Bermuda, Barbados, Brunei, Botswana
Появились в панели позже 2018 года: 24 стран


**Что видим.** Экспорт российской фанеры (HS4 4412) по BACI за 2018–2024 годы: 141 страна, 701 строка. Дубликатов пары страна–год нет. Нулевых потоков тоже нет: BACI хранит только положительную торговлю, так что «нет строки» значит «поток не зафиксирован». Нули я не дописываю, их всё равно нельзя прологарифмировать. Панель несбалансированная: все 7 лет есть только у 48 стран, у 11 стран одно наблюдение. Главная причина — 2022 год. Число стран падает со 120 в 2021 году до 53 в 2024-м, у 68 стран последний поток в 2022 году или раньше (Австралия, Бельгия, Болгария и другие страны ЕС). Это совпадает с европейским запретом на российскую древесину. Ещё 24 страны появились в панели позже 2018 года. Экспорт упал с пиковых 2 005 млн долл. в 2021 году до 731 млн в 2024-м.

### **[Синий, 1 балл]** Пункт 1б. Итоговая панель country×year

Присоедини ВВП (`NY.GDP.MKTP.CD`), население (`SP.POP.TOTL`) и обменный курс (`PA.NUS.FCRF`) из World Bank.


In [3]:
# Присоедини ВВП, население и обменный курс из World Bank.
panel = trade_panel.copy()
WDI_INDICATORS = {
    "NY.GDP.MKTP.CD": "GDP",
    "SP.POP.TOTL": "Population",
    "PA.NUS.FCRF": "ER",
}
wdi_urls = {}
for indicator, column in WDI_INDICATORS.items():
    panel, wdi_urls[indicator] = attach_wdi(panel, indicator, column, START_YEAR, END_YEAR)

# Года WDI берём ровно те же, что и у торговли: склейка идёт по паре ISO3–Year
missing = panel[list(WDI_INDICATORS.values())].isna().sum().rename("Пропусков")
display(missing.to_frame())
no_factor = panel.loc[panel[list(WDI_INDICATORS.values())].isna().any(axis=1)]
print("Строки без факторов:", no_factor.groupby("Страна").size().to_dict())

# Сформируй общую выборку для всех моделей и создай логарифмы.
# 1) убираю строки без ВВП, населения или курса; 2) убираю страны с одним годом:
# в FE они ничего не дают, а выборка должна быть одной для всех моделей
analysis_panel = panel.dropna(subset=list(WDI_INDICATORS.values())).copy()
obs_per_country = analysis_panel.groupby("ISO3")["Year"].transform("size")
singletons = analysis_panel.loc[obs_per_country == 1, "Страна"].tolist()
analysis_panel = analysis_panel.loc[obs_per_country > 1].copy()
print(f"Убраны страны с одним наблюдением ({len(singletons)}): {', '.join(singletons)}")

analysis_panel["ln_Exports"] = np.log(analysis_panel["Exports"])
analysis_panel["ln_GDP"] = np.log(analysis_panel["GDP"])
analysis_panel["ln_Population"] = np.log(analysis_panel["Population"])
analysis_panel = analysis_panel.sort_values(["ISO3", "Year"]).reset_index(drop=True)

print(f"Общая выборка: {len(analysis_panel)} наблюдений, {analysis_panel['ISO3'].nunique()} стран, "
      f"{analysis_panel['Year'].min()}–{analysis_panel['Year'].max()}")
print("Единицы: Exports и GDP — текущие доллары США, Population — человек, "
      "ER — единиц национальной валюты за 1 доллар (среднее за год)")
display(analysis_panel[["Exports", "GDP", "Population", "ER"]].describe().T)


,Пропусков
GDP,11
Population,7
ER,24


Строки без факторов: {'Afghanistan': 1, 'British Virgin Islands': 1, 'Chinese Taipei': 7, 'Cuba': 5, 'Guinea': 1, 'North Korea': 1, 'Palestine': 4, 'Turkmenistan': 4}
Убраны страны с одним наблюдением (8): Bangladesh, Brunei, Botswana, Central African Republic, Haiti, Saint Lucia, Mozambique, Nepal
Общая выборка: 669 наблюдений, 126 стран, 2018–2024
Единицы: Exports и GDP — текущие доллары США, Population — человек, ER — единиц национальной валюты за 1 доллар (среднее за год)


,count,mean,std,min,25%,50%,75%,max
Exports,669.000,"13,328,981.655","33,890,661.124",59.000,"173,821.000","1,550,703.000","11,147,443.000","433,163,766.000"
GDP,669.000,"911,065,683,846.147","3,078,052,011,973.668","1,487,173,794.805","39,667,757,527.658","135,874,093,203.180","480,466,363,176.806","29,298,013,000,000.000"
Population,669.000,"63,585,314.716","205,416,558.731","63,831.000","4,500,424.000","10,486,941.000","44,761,099.000","1,450,935,791.000"
ER,669.000,"1,225.461","5,750.795",0.302,1.000,7.697,108.650,"89,500.000"


**Что видим.** ВВП (NY.GDP.MKTP.CD, текущие доллары), население (SP.POP.TOTL, человек) и курс (PA.NUS.FCRF, единиц нацвалюты за доллар, среднее за год) беру из World Bank ровно за 2018–2024 годы. Склейка идёт по паре страна–год, так что года торговли и факторов совпадают, ничего не подставляю из соседних лет. Факторов нет у 8 стран, в основном нет курса: Тайвань (7 строк), Куба (5), Палестина и Туркменистан (по 4) и ещё четыре страны по одной строке. Эти строки убираю. Ещё убираю 8 стран, которые после этого остались с одним годом: фиксированный эффект для них съедает всё наблюдение. Итоговая общая выборка для всех трёх моделей — 669 наблюдений, 126 стран. Курс в уровнях очень разный: медиана 7,7, максимум 89 500 за доллар.

### **[Синий, 1.5 балла]** Пункт 1в. Оценка pooled OLS

$$\ln(\text{Exports}_{it}) = b_0 + b_1\ln(\text{GDP}_{it}) + b_2\ln(\text{Population}_{it}) + b_3\,\text{ER}_{it} + u_{it}$$


In [4]:
# Оцени pooled OLS с робастными стандартными ошибками (HC1).
FEATURES = ["ln_GDP", "ln_Population", "ER"]
X_pooled = sm.add_constant(analysis_panel[FEATURES])
pooled_res = sm.OLS(analysis_panel["ln_Exports"], X_pooled).fit(cov_type="HC1")

# Выведи коэффициенты, se, p-value, R² и N.
pooled_table = pd.DataFrame({
    "Коэффициент": pooled_res.params,
    "Робастная se": pooled_res.bse,
    "p-value": pooled_res.pvalues,
})
# У курса коэффициент очень маленький, поэтому выводим больше знаков
with pd.option_context("display.float_format", "{:.4g}".format):
    display(pooled_table)
print(f"R2 = {pooled_res.rsquared:.3f}, N = {int(pooled_res.nobs)}")

er_effect = pooled_res.params["ER"] * 1000
print(f"Рост курса на 1000 единиц валюты за доллар: ln экспорта выше на {er_effect:.3f}")


,Коэффициент,Робастная se,p-value
const,-5.191,1.274,4.593e-05
ln_GDP,1.003,0.08852,9.355e-30
ln_Population,-0.4057,0.09201,1.035e-05
ER,3.215e-05,8.719e-06,0.0002266


R2 = 0.253, N = 669
Рост курса на 1000 единиц валюты за доллар: ln экспорта выше на 0.032


**Что видим.** $\ln Exports = -5{,}19 + 1{,}003\,\ln GDP - 0{,}406\,\ln Pop + 0{,}0000322\,ER$. Робастные se: 0,089, 0,092 и 0,0000087. Все три коэффициента значимы на 1%. $R^2 = 0{,}253$, $N = 669$. ВВП больше на 1% — экспорт фанеры в среднем больше на 1%. Население больше на 1% при том же ВВП — экспорт меньше на 0,41%. Курс выше на 1000 единиц за доллар — экспорт выше примерно на 3,2%.

### **[Синий, 1.5 балла]** Пункт 1г. Интерпретация pooled OLS

Проинтерпретируй знак, экономический смысл и статистическую значимость каждого коэффициента.


In [5]:
# Если в pooled OLS есть неожиданный результат, сформулируй проверяемую гипотезу
# и проверь её хотя бы одним простым способом на тех же данных.

# Неожиданно: при том же ВВП население со знаком минус.
# Гипотеза: это эффект дохода на душу. Если ВВП тот же, а людей больше, страна беднее,
# а фанера — товар для строительства и мебели, его больше покупают богатые страны.
check = analysis_panel.copy()
check["ln_GDP_pc"] = check["ln_GDP"] - check["ln_Population"]
print(f"Корреляция ln населения и ln ВВП на душу: {check['ln_Population'].corr(check['ln_GDP_pc']):.3f}")

# Проверка 1: терцили по ВВП на душу при похожем общем ВВП. Делю страны на половины
# по ln ВВП и внутри каждой смотрю на терцили дохода на душу
check["Половина по ВВП"] = pd.qcut(check["ln_GDP"], 2, labels=["меньший ВВП", "больший ВВП"])
check["Терциль дохода"] = check.groupby("Половина по ВВП", observed=True)["ln_GDP_pc"].transform(
    lambda values: pd.qcut(values, 3, labels=False) + 1
)
income_check = check.groupby(["Половина по ВВП", "Терциль дохода"], observed=True).agg(
    Наблюдений=("ln_Exports", "size"),
    Средний_ln_экспорта=("ln_Exports", "mean"),
    Медианный_экспорт_млн=("Exports", lambda values: values.median() / 1e6),
    ВВП_на_душу_тыс=("ln_GDP_pc", lambda values: np.exp(values).median() / 1000),
)
display(income_check)

# Проверка 2: та же регрессия, но вместо ln населения — ln ВВП на душу
X_pc = sm.add_constant(check[["ln_GDP", "ln_GDP_pc", "ER"]])
pc_res = sm.OLS(check["ln_Exports"], X_pc).fit(cov_type="HC1")
with pd.option_context("display.float_format", "{:.4g}".format):
    display(pd.DataFrame({"Коэффициент": pc_res.params, "p-value": pc_res.pvalues}))

# Второе, что удивляет: курс в уровнях значим и с плюсом. Смотрю, кто стоит
# в верхнем дециле курса
top_er = (
    check.groupby(["ISO3", "Страна"])
    .agg(Курс=("ER", "mean"), Экспорт_млн=("Exports", lambda values: values.mean() / 1e6))
    .sort_values("Курс", ascending=False)
    .head(10)
)
display(top_er)


Корреляция ln населения и ln ВВП на душу: -0.292


Наблюдений  Средний_ln_экспорта  \
Половина по ВВП Терциль дохода                                    
меньший ВВП     1                      112               12.156   
                2                      111               13.058   
                3                      112               13.458   
больший ВВП     1                      112               14.243   
                2                      111               15.703   
                3                      111               15.503   

                                Медианный_экспорт_млн  ВВП_на_душу_тыс  
Половина по ВВП Терциль дохода                                          
меньший ВВП     1                               0.155            2.140  
                2                               0.370            6.719  
                3                               0.724           22.153  
больший ВВП     1                               1.430            4.901  
                2                              12.706           28.282  
                3                               8.983           54.848

,Коэффициент,p-value
const,-5.191,4.593e-05
ln_GDP,0.5972,1.231e-34
ln_GDP_pc,0.4057,1.035e-05
ER,3.215e-05,0.0002266


,,Курс,Экспорт_млн
ISO3,Страна,,
IRN,Iran,"41,772.866",6.946
VNM,Vietnam,"23,179.829",7.894
LBN,Lebanon,"15,844.732",2.023
IDN,Indonesia,"14,745.306",0.671
UZB,Uzbekistan,"10,429.626",22.518
PRY,Paraguay,"6,764.280",0.221
KHM,Cambodia,"4,084.218",0.159
COL,Colombia,"3,761.633",0.434
UGA,Uganda,"3,699.381",0.038


**Что видим.** Неожиданный знак у населения. Моя гипотеза: при том же ВВП больше людей значит беднее страна, а фанеру (стройка, мебель) больше берут богатые. Корреляция ln населения с ln ВВП на душу равна −0,29. Проверка терцилями: в обеих половинах по ВВП средний ln экспорта растёт от бедного терциля к богатому (12,2 → 13,1 → 13,5 и 14,2 → 15,7 → 15,5). Медианный экспорт в меньших экономиках растёт с 0,16 до 0,72 млн долл. Если заменить ln населения на ln ВВП на душу, коэффициент становится +0,41 с тем же p-value: это та же модель, просто иначе записанная. Гипотеза подтверждается: перспективнее страны с высоким доходом на душу, а не просто многолюдные. Второй сюрприз — курс в уровнях. Наверху Иран (41 773), Вьетнам, Ливан, Индонезия, Узбекистан. Коэффициент говорит скорее о том, в какой валюте считает страна, а не о девальвации. Сравнивать курсы разных валют в уровнях нельзя.

## **[Красный, 2 балла]** Задание 2. Модель с фиксированными эффектами

### **[Красный, 0.5 балла]** Пункт 2а. Экономическая мотивация

Какие ненаблюдаемые характеристики твоей страны-импортёра могут быть постоянными во времени? Почему они могут коррелировать с ВВП, населением или курсом?


### **[Красный, 1 балл]** Пункт 2б. Оценка FE-модели

Оцени модель с фиксированными эффектами по странам и годам. Выбери свой постоянный признак (участие в союзе, общая граница и так далее), попробуй добавить его в FE и объясни результат.


In [6]:
# Оцени FE-модель со страновыми и годовыми эффектами.
panel_idx = analysis_panel.set_index(["ISO3", "Year"])
fe_res = PanelOLS(
    panel_idx["ln_Exports"],
    panel_idx[FEATURES],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

fe_table = pd.DataFrame({
    "Коэффициент": fe_res.params,
    "Кластерная se": fe_res.std_errors,
    "p-value": fe_res.pvalues,
})
with pd.option_context("display.float_format", "{:.4g}".format):
    display(fe_table)
print(f"R2 within = {fe_res.rsquared_within:.3f}, N = {fe_res.nobs}, "
      f"стран = {fe_res.entity_info['total']:.0f}")

# Сколько изменчивости остаётся внутри стран
within_share = {}
for column in ["ln_Exports", *FEATURES]:
    within = analysis_panel[column] - analysis_panel.groupby("ISO3")[column].transform("mean")
    within_share[column] = within.var() / analysis_panel[column].var()
print("Доля дисперсии внутри стран:", {k: round(float(v), 4) for k, v in within_share.items()})


,Коэффициент,Кластерная se,p-value
ln_GDP,1.016,0.7482,0.1751
ln_Population,4.672,3.643,0.2002
ER,2.805e-05,6.836e-06,4.717e-05


R2 within = -0.090, N = 669, стран = 126
Доля дисперсии внутри стран: {'ln_Exports': 0.1871, 'ln_GDP': 0.005, 'ln_Population': 0.0002, 'ER': 0.2934}


**Что видим.** С фиксированными эффектами по странам и годам: $\hat b_1 = 1{,}016$ (se 0,748, p = 0,175), $\hat b_2 = 4{,}672$ (se 3,643, p = 0,200), $\hat b_3 = 0{,}000028$ (p < 0,001). Значим только курс. Причина видна в долях дисперсии внутри стран: у ln ВВП это 0,5%, у ln населения 0,02%. После вычитания среднего по стране от этих переменных почти ничего не остаётся, и оценки получаются очень неточными. У курса внутри стран 29% дисперсии (Турция, Египет, Иран девальвировались), поэтому он остался значимым. $R^2$ within = −0,09: сверх общих годовых шоков регрессоры почти не объясняют изменений внутри стран.

In [7]:
# Создай выбранный постоянный признак страны и попробуй добавить его в FE.
# Признак: страна — член ЕАЭС вместе с Россией (Армения, Беларусь, Казахстан, Киргизия)
EAEU = ["ARM", "BLR", "KAZ", "KGZ"]
analysis_panel["EAEU"] = analysis_panel["ISO3"].isin(EAEU).astype(float)
panel_idx = analysis_panel.set_index(["ISO3", "Year"])
print("Стран ЕАЭС в выборке:", analysis_panel.loc[analysis_panel["EAEU"].eq(1), "Страна"].unique().tolist())
print("Максимальный разброс EAEU внутри страны:",
      analysis_panel.groupby("ISO3")["EAEU"].std().max())

FE_FEATURES_EAEU = [*FEATURES, "EAEU"]
try:
    PanelOLS(
        panel_idx["ln_Exports"], panel_idx[FE_FEATURES_EAEU],
        entity_effects=True, time_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)
except Exception as error:
    print(f"Ошибка: {type(error).__name__}")
    print(str(error)[:400])

# Если разрешить выкидывать поглощённые переменные, EAEU просто пропадёт из модели
fe_eaeu = PanelOLS(
    panel_idx["ln_Exports"], panel_idx[FE_FEATURES_EAEU],
    entity_effects=True, time_effects=True, drop_absorbed=True,
).fit(cov_type="clustered", cluster_entity=True)
print("Оценённые параметры:", list(fe_eaeu.params.index))


Стран ЕАЭС в выборке: ['Armenia', 'Belarus', 'Kazakhstan', 'Kyrgyzstan']
Максимальный разброс EAEU внутри страны: 0.0
Ошибка: AbsorbingEffectError

The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          EAEU

Set drop_absorbed=True t
Оценённые параметры: ['ln_GDP', 'ln_Population', 'ER']


**Что видим.** Постоянный признак — членство в ЕАЭС (Армения, Беларусь, Казахстан, Киргизия, все четыре в выборке). Внутри каждой страны он не меняется (стандартное отклонение 0). После вычитания странового среднего он превращается в столбец нулей и полностью совпадает со страновым эффектом. Поэтому `PanelOLS` выдаёт `AbsorbingEffectError`, а с `drop_absorbed=True` просто выкидывает EAEU. Это не ошибка в данных, FE в принципе не может оценить такой признак.

### **[Красный, 0.5 балла]** Пункт 2в. Сравнение с pooled OLS

Сравни коэффициенты и объясни направление изменений.


In [8]:
# Сопоставь коэффициенты pooled OLS и FE в одной таблице.
comparison = pd.DataFrame({
    "Pooled OLS": pooled_res.params[FEATURES],
    "p pooled": pooled_res.pvalues[FEATURES],
    "FE": fe_res.params[FEATURES],
    "p FE": fe_res.pvalues[FEATURES],
})
comparison["FE − pooled"] = comparison["FE"] - comparison["Pooled OLS"]
with pd.option_context("display.float_format", "{:.4g}".format):
    display(comparison)
print(f"N: pooled {int(pooled_res.nobs)}, FE {fe_res.nobs} — выборка одна и та же")


,Pooled OLS,p pooled,FE,p FE,FE − pooled
ln_GDP,1.003,9.355e-30,1.016,0.1751,0.01306
ln_Population,-0.4057,1.035e-05,4.672,0.2002,5.077
ER,3.215e-05,0.0002266,2.805e-05,4.717e-05,-4.104e-06


N: pooled 669, FE 669 — выборка одна и та же


**Что видим.** Сильнее всего изменилось население: с −0,41 (значимо) до +4,67 (незначимо). В pooled OLS этот коэффициент ловил разницу между странами — богатые против бедных. В FE остаются только изменения населения внутри страны за 7 лет, они крошечные, и оценка ничего не значит. ВВП почти не сдвинулся (1,003 → 1,016), но потерял значимость (p с почти нуля до 0,175) по той же причине. Курс почти тот же (0,0000322 → 0,0000281) и остался значимым. Обе модели оценены на одних и тех же 669 наблюдениях.

## **[Чёрный, 3 балла + 1 бонусный]** Задание 3. RE, тест Хаусмана и итоговая записка

### **[Чёрный, 1 балл]** Пункт 3а. Оценка RE-модели

Оцени модель со случайными эффектами, включая твой постоянный признак и годовые фиктивные переменные.


In [9]:
# Оцени RE-модель с постоянным признаком и годовыми фиктивными переменными.
year_dummies = pd.get_dummies(panel_idx.index.get_level_values("Year"), prefix="Year",
                              drop_first=True, dtype=float)
year_dummies.index = panel_idx.index
RE_FEATURES = [*FEATURES, "EAEU"]
X_re = sm.add_constant(pd.concat([panel_idx[RE_FEATURES], year_dummies], axis=1))
re_res = RandomEffects(panel_idx["ln_Exports"], X_re).fit(cov_type="clustered", cluster_entity=True)

re_table = pd.DataFrame({
    "Коэффициент": re_res.params,
    "Кластерная se": re_res.std_errors,
    "p-value": re_res.pvalues,
}).loc[["const", *RE_FEATURES]]
with pd.option_context("display.float_format", "{:.4g}".format):
    display(re_table)
print(f"R2 overall = {re_res.rsquared_overall:.3f}, N = {re_res.nobs}")
print("Доля дисперсии ошибки, приходящаяся на страновой эффект:",
      round(float(re_res.variance_decomposition["Percent due to Effects"]), 3))
eaeu_ratio = np.exp(re_res.params["EAEU"])
print(f"exp(коэффициент EAEU) = {eaeu_ratio:.2f}")


,Коэффициент,Кластерная se,p-value
const,-10.28,2.342,1.314e-05
ln_GDP,1.302,0.1526,9.805e-17
ln_Population,-0.575,0.1647,0.0005147
ER,3.206e-05,6.308e-06,4.869e-07
EAEU,3.966,0.4588,4.092e-17


R2 overall = 0.342, N = 669
Доля дисперсии ошибки, приходящаяся на страновой эффект: 0.729
exp(коэффициент EAEU) = 52.78


**Что видим.** RE с годовыми фиктивными: ln ВВП 1,302, ln населения −0,575, курс 0,000032, ЕАЭС 3,966. Всё значимо на 1%, $R^2$ overall = 0,342. В отличие от FE, здесь признак ЕАЭС оценивается: при тех же ВВП, населении и курсе экспорт в страну ЕАЭС примерно в $e^{3{,}97} \approx 53$ раза больше. 72,9% дисперсии ошибки приходится на страновой эффект, то есть постоянные различия между странами очень сильные.

### Дополнительная идея — исключительно по желанию, без баллов

Если хочется расширить работу, проверь устойчивость результатов к другому описанию рынков. Например, добавь расстояние до России или регион страны, объедини страны в более крупные группы либо попробуй кластеризацию по характеристикам рынка и модельным разрывам.

Это не обязательный пункт и не влияет на оценку. Смысл расширения — проверить, сохраняются ли основные выводы при другой единице наблюдения или дополнительной информации о географии рынка.


### **[Чёрный, 1 балл]** Пункт 3б. Тест Хаусмана

Сформулируй $H_0$ и $H_1$ математически, приведи статистику, степени свободы и $p$-значение, сделай вывод.


In [10]:
# Проведи тест Хаусмана по общим коэффициентам FE и RE.
# H0: Cov(a_i, X_it) = 0 — страновой эффект не связан с регрессорами, RE и FE состоятельны,
#     RE эффективнее. H1: Cov(a_i, X_it) != 0 — состоятельна только FE.
# H = (b_FE − b_RE)' [Var(b_FE) − Var(b_RE)]^(-1) (b_FE − b_RE) ~ chi2(k), k = 3
# Для классического теста беру обычные (неробастные) ковариации обеих моделей
fe_plain = PanelOLS(panel_idx["ln_Exports"], panel_idx[FEATURES],
                    entity_effects=True, time_effects=True).fit()
re_plain = RandomEffects(panel_idx["ln_Exports"], X_re).fit()

b_diff = (fe_plain.params[FEATURES] - re_plain.params[FEATURES]).to_numpy()
v_diff = (fe_plain.cov.loc[FEATURES, FEATURES] - re_plain.cov.loc[FEATURES, FEATURES]).to_numpy()
print("Собственные числа Var(b_FE) − Var(b_RE):", np.linalg.eigvalsh(v_diff))

H = float(b_diff @ np.linalg.inv(v_diff) @ b_diff)
df = len(FEATURES)
p_value = stats.chi2.sf(H, df)
print(f"H = {H:.3f}, df = {df}, p-value = {p_value:.4f}")
print(f"Критическое значение chi2({df}) на 5%: {stats.chi2.ppf(0.95, df):.3f}")

hausman_table = pd.DataFrame({
    "FE": fe_plain.params[FEATURES],
    "RE": re_plain.params[FEATURES],
    "Разность": b_diff,
    "Var FE − Var RE": np.diag(v_diff),
})
with pd.option_context("display.float_format", "{:.4g}".format):
    display(hausman_table)

final_name = "FE" if p_value < 0.05 else "RE"
print(f"Модель для пункта 3г: {final_name}")


Собственные числа Var(b_FE) − Var(b_RE): [2.83094005e-11 2.60598538e-01 6.33851098e+00]
H = 5.114, df = 3, p-value = 0.1636
Критическое значение chi2(3) на 5%: 7.815


,FE,RE,Разность,Var FE − Var RE
ln_GDP,1.016,1.302,-0.2861,0.2624
ln_Population,4.672,-0.575,5.247,6.337
ER,2.805e-05,3.206e-05,-4.012e-06,3.571e-11


Модель для пункта 3г: RE


**Что видим.** $H_0:\ \mathrm{Cov}(a_i, X_{it}) = 0$ (RE и FE состоятельны, RE эффективнее). $H_1:\ \mathrm{Cov}(a_i, X_{it}) \ne 0$ (состоятельна только FE). Статистика $H = (\hat b_{FE}-\hat b_{RE})'[\hat V_{FE}-\hat V_{RE}]^{-1}(\hat b_{FE}-\hat b_{RE}) = 5{,}114$ при $df = 3$, p = 0,164. Критическое значение $\chi^2_3$ на 5% равно 7,815, так что $H_0$ не отвергаем и для пункта 3г беру RE. Разности большие (у населения 5,25), но и дисперсии FE огромные, поэтому тест слабый. Он говорит «нет оснований отвергнуть», а не «эффекты точно не коррелируют». Все собственные числа разности ковариаций положительные, обратная матрица считается нормально.

### **[Чёрный, 1 балл]** Пункт 3в. Итоговая аналитическая записка

Сопоставь pooled OLS, FE и RE, выбери предпочтительную спецификацию, укажи ограничения.


In [11]:
# Сравни in-sample подгонку pooled OLS, FE и RE.
# Важно: для FE/RE полный fitted включает fitted_values + estimated_effects.
fitted = pd.DataFrame(index=panel_idx.index)
fitted["Факт"] = panel_idx["ln_Exports"]
fitted["Pooled OLS"] = pooled_res.fittedvalues.to_numpy()
fitted["FE"] = fe_res.fitted_values["fitted_values"] + fe_res.estimated_effects["estimated_effects"]
fitted["RE"] = re_res.fitted_values["fitted_values"] + re_res.estimated_effects["estimated_effects"]

# Рассчитай MAE и RMSE и собери таблицу сравнения.
fit_rows = []
for model in ["Pooled OLS", "FE", "RE"]:
    error_log = fitted["Факт"] - fitted[model]
    error_money = (np.exp(fitted["Факт"]) - np.exp(fitted[model])) / 1e6
    fit_rows.append({
        "Модель": model,
        "N": len(error_log),
        "MAE, ln": error_log.abs().mean(),
        "RMSE, ln": np.sqrt((error_log ** 2).mean()),
        "R2, ln": 1 - (error_log ** 2).sum() / ((fitted["Факт"] - fitted["Факт"].mean()) ** 2).sum(),
        "MAE, млн долл": error_money.abs().mean(),
        "RMSE, млн долл": np.sqrt((error_money ** 2).mean()),
    })
fit_comparison = pd.DataFrame(fit_rows).set_index("Модель")
display(fit_comparison)


,N,"MAE, ln","RMSE, ln","R2, ln","MAE, млн долл","RMSE, млн долл"
Модель,,,,,,
Pooled OLS,669,1.884,2.396,0.253,11.629,31.926
FE,669,0.720,1.076,0.849,6.483,20.068
RE,669,0.832,1.198,0.813,7.014,19.908


**Что видим.** Все три модели на одних и тех же 669 наблюдениях. Pooled OLS подгоняет хуже всех: $R^2$ в логарифмах 0,253, MAE 1,88. FE и RE с учётом страновых эффектов — 0,849 и 0,813, MAE 0,72 и 0,83. В деньгах MAE 11,6, 6,5 и 7,0 млн долл. По RMSE в деньгах FE и RE почти равны (20,1 и 19,9). Почти весь выигрыш даёт страновой эффект, а не ВВП и население. FE подгоняет чуть лучше, потому что эффекты ничем не ограничены, но для 3г я беру RE: так решил тест Хаусмана.

### **[Чёрный, 1 бонусный балл]** Пункт 3г. Топ-10 недоосвоенных рынков

Используя финальную модель (ту, что выбрана по тесту Хаусмана), найди 10 стран с наибольшим положительным разрывом $G_{it}=\hat y_{it}-y_{it}$ в последнем году.


In [12]:
# Используя модель, выбранную по тесту Хаусмана, построй Top-10.
# Верни fitted из логарифмов в денежную шкалу, затем рассчитай
# G = y_hat - y и g = G / y_hat для последнего года.
LAST_YEAR = analysis_panel["Year"].max()
gaps = analysis_panel[["ISO3", "Страна", "Year", "Exports", "EAEU"]].copy()
gaps["y_hat"] = np.exp(fitted[final_name].to_numpy())
gaps = gaps.loc[gaps["Year"].eq(LAST_YEAR)].copy()

gaps["y, млн долл"] = gaps["Exports"] / 1e6
gaps["y_hat, млн долл"] = gaps["y_hat"] / 1e6
gaps["G, млн долл"] = gaps["y_hat, млн долл"] - gaps["y, млн долл"]
gaps["g"] = gaps["G, млн долл"] / gaps["y_hat, млн долл"]

print(f"Модель {final_name}, {LAST_YEAR} год: стран {len(gaps)}, "
      f"с положительным разрывом {(gaps['G, млн долл'] > 0).sum()}")
top10 = gaps.loc[gaps["G, млн долл"] > 0].nlargest(10, "G, млн долл")
display(top10[["ISO3", "Страна", "y, млн долл", "y_hat, млн долл", "G, млн долл", "g"]]
        .reset_index(drop=True))
print(f"Сумма разрывов Top-10: {top10['G, млн долл'].sum():.1f} млн долл, "
      f"факт по ним: {top10['y, млн долл'].sum():.1f} млн долл")

# Почти весь список — страны ЕС, где ввоз российской древесины и фанеры запрещён с 2022 года.
# Помечаю их и отдельно строю Top-10 без ЕС
EU27 = ["AUT", "BEL", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN",
        "IRL", "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU", "SVK", "SVN", "ESP", "SWE"]
gaps["ЕС"] = gaps["ISO3"].isin(EU27)
print(f"Стран ЕС в Top-10: {top10['ISO3'].isin(EU27).sum()}")
top10_non_eu = gaps.loc[(gaps["G, млн долл"] > 0) & ~gaps["ЕС"]].nlargest(10, "G, млн долл")
display(top10_non_eu[["ISO3", "Страна", "y, млн долл", "y_hat, млн долл", "G, млн долл", "g"]]
        .reset_index(drop=True))

# Для сравнения: разрыв по pooled OLS у тех же стран (без странового эффекта)
pooled_last = np.exp(fitted.loc[(slice(None), LAST_YEAR), "Pooled OLS"]).droplevel("Year") / 1e6
top10_check = top10.set_index("ISO3")[["Страна", "y, млн долл"]].copy()
top10_check["y_hat pooled, млн долл"] = pooled_last.reindex(top10_check.index)
display(top10_check)


Модель RE, 2024 год: стран 52, с положительным разрывом 19


,ISO3,Страна,"y, млн долл","y_hat, млн долл","G, млн долл",g
0,DEU,Germany,0.490,8.823,8.333,0.944
1,FRA,France,0.559,2.390,1.832,0.766
2,EST,Estonia,0.112,1.465,1.353,0.923
3,AUT,Austria,0.426,1.637,1.211,0.740
4,ESP,Spain,0.190,1.362,1.172,0.861
5,CZE,Czechia,0.000,0.817,0.817,0.999
6,LBN,Lebanon,2.499,3.026,0.528,0.174
7,CHE,Switzerland,0.002,0.439,0.436,0.995
8,LUX,Luxembourg,0.015,0.268,0.252,0.942
9,JPN,Japan,1.463,1.704,0.241,0.141


Сумма разрывов Top-10: 16.2 млн долл, факт по ним: 5.8 млн долл
Стран ЕС в Top-10: 7


,ISO3,Страна,"y, млн долл","y_hat, млн долл","G, млн долл",g
0,LBN,Lebanon,2.499,3.026,0.528,0.174
1,CHE,Switzerland,0.002,0.439,0.436,0.995
2,JPN,Japan,1.463,1.704,0.241,0.141
3,SRB,Serbia,1.034,1.265,0.231,0.183
4,COL,Colombia,0.129,0.185,0.056,0.302
5,MNE,Montenegro,0.045,0.097,0.052,0.537
6,BHR,Bahrain,0.006,0.057,0.051,0.890
7,MKD,North Macedonia,0.011,0.040,0.029,0.729
8,BIH,Bosnia and Herzegovina,0.174,0.200,0.026,0.131
9,BRA,Brazil,0.030,0.051,0.021,0.405


,Страна,"y, млн долл","y_hat pooled, млн долл"
ISO3,,,
DEU,Germany,0.490,17.362
FRA,France,0.559,12.673
EST,Estonia,0.112,0.835
AUT,Austria,0.426,4.823
ESP,Spain,0.190,7.926
CZE,Czechia,0.000,2.917
LBN,Lebanon,2.499,4.968
CHE,Switzerland,0.002,8.831
LUX,Luxembourg,0.015,2.410


**Что видим.** По RE в 2024 году положительный разрыв есть только у 19 стран из 52. Первые места — Германия (факт 0,49 млн долл., ориентир 8,82 млн, $G = 8{,}3$ млн), Франция, Эстония, Австрия, Испания, Чехия. 7 из 10 стран Top-10 — ЕС. Их страновой эффект оценён в основном на 2018–2021 годах, когда они много покупали, а с 2022 года ввоз российской фанеры туда запрещён. Так что разрыв здесь показывает санкции, а не недоосвоенный спрос. Без ЕС разрывы маленькие: Ливан 0,53 млн, Швейцария 0,44, Япония 0,24, Сербия 0,23 млн. Наши Корея, Индия и Саудовская Аравия в Top-10 не попали, то есть по модели они уже покупают на своём уровне. Pooled OLS без странового эффекта рисует гораздо большие разрывы (Германия 17,4 млн, Япония 13,3 млн против факта 1,5 млн). Разрыв — это отклонение от модельного ориентира, а не объём, который можно продать: нужно проверить доступ к рынку, пошлины, логистику и цены.